# Task 1 — Supervised Learning

## Bank Marketing Subscription Prediction

**Business objective:** Rank customers by predicted subscription probability so the bank can prioritize the 500 customers it can contact.

**Decision point:** The prediction must use only information available before the current call. Therefore, `duration` is excluded because it is known only after the call.

## 1. Imports

In [171]:
import os
import csv
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix
)

from sklearn.model_selection import TimeSeriesSplit, GridSearchCV

RANDOM_STATE = 42

## 2. Load the Dataset

In [172]:
df = pd.read_csv(
    "bank-additional-full.csv",
    sep=";",
    quoting=csv.QUOTE_NONE
)

df = df.map(
    lambda x: x.strip('"') if isinstance(x, str) else x
)
df.columns = df.columns.str.strip('"')

df["age"] = pd.to_numeric(df["age"], errors="coerce")

print("Shape:", df.shape)
display(df.head())

Shape: (41188, 21)


,age,job,marital,education,default,housing,loan,contact,month,day_of_week,...,campaign,pdays,previous,poutcome,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,y
0,56,housemaid,married,basic.4y,no,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
1,57,services,married,high.school,unknown,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
2,37,services,married,high.school,no,yes,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
3,40,admin.,married,basic.6y,no,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
4,56,services,married,high.school,no,no,yes,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no


## 3. Define Features and Target

`y` is the target, so it is not an input feature.

`duration` is excluded because it is post-call information and would cause information leakage.

In [173]:
X = df.drop(columns=["y", "duration"])
y = df["y"]

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (41188, 19)
y shape: (41188,)


## 4. Temporal Train / Holdout Split

In [174]:
split_index = int(len(df) * 0.80)

X_train = X.iloc[:split_index].copy()
X_holdout = X.iloc[split_index:].copy()

y_train = y.iloc[:split_index].copy()
y_holdout = y.iloc[split_index:].copy()

print("Training rows:", len(X_train))
print("Holdout rows:", len(X_holdout))

print("\nTraining subscription rate:")
print((y_train == "yes").mean())

print("\nHoldout subscription rate:")
print((y_holdout == "yes").mean())


Training rows: 32950
Holdout rows: 8238

Training subscription rate:
0.0637329286798179

Holdout subscription rate:
0.3083272638990046


### Temporal Target Distribution Shift

The temporal holdout has a substantially different subscription rate from the training period. The training period has a subscription rate of approximately 6.37%, while the later holdout period has a rate of approximately 30.83%.

This indicates a strong temporal shift in the target distribution. Therefore, model performance on the holdout should be interpreted as performance on a later time period rather than as performance on a randomly sampled test set.

## 5. Identify Numerical and Categorical Features

In [175]:
numeric_features = X_train.select_dtypes(include=np.number).columns.tolist()
categorical_features = X_train.select_dtypes(include="object").columns.tolist()

print("Numerical features:", numeric_features)
print("\nCategorical features:", categorical_features)

Numerical features: ['age', 'campaign', 'pdays', 'previous', 'emp.var.rate', 'cons.price.idx', 'cons.conf.idx', 'euribor3m', 'nr.employed']

Categorical features: ['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'day_of_week', 'poutcome']


## 6. Preprocessing Pipeline

- Numerical features: median imputation + standardization
- Categorical features: most-frequent imputation + one-hot encoding
- Categorical values recorded as `unknown` are retained as an explicit category rather than automatically converted to missing values. This allows the model to learn whether the `unknown` state itself contains predictive information.
The preprocessing is fitted through the Pipeline using training data only.

In [176]:
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features)
])

## 7. Baseline Model

In [177]:
baseline = DummyClassifier(strategy="most_frequent")

baseline.fit(X_train, y_train)

baseline_pred = baseline.predict(X_holdout)
baseline_prob = baseline.predict_proba(X_holdout)[:, 1]


y_holdout_binary = (y_holdout == "yes").astype(int)

print("Baseline accuracy:", accuracy_score(y_holdout_binary, (baseline_pred == "yes").astype(int)))
print("Baseline ROC-AUC:", roc_auc_score(y_holdout_binary, baseline_prob))

Baseline accuracy: 0.6916727361009953
Baseline ROC-AUC: 0.5


## 8. Evaluation Functions

We will evaluate both ordinary classification performance and the ranking performance that matters for the 500-customer constraint.

In [178]:
def evaluate_model(name, model, X_test, y_test):
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    y_true_binary = (y_test == "yes").astype(int)
    y_pred_binary = (y_pred == "yes").astype(int)

    return {
        "model": name,
        "accuracy": accuracy_score(y_true_binary, y_pred_binary),
        "precision": precision_score(y_true_binary, y_pred_binary, zero_division=0),
        "recall": recall_score(y_true_binary, y_pred_binary, zero_division=0),
        "f1": f1_score(y_true_binary, y_pred_binary, zero_division=0),
        "roc_auc": roc_auc_score(y_true_binary, y_prob)
    }


def precision_at_k(y_true, y_prob, k=500):
    y_true = y_true.reset_index(drop=True)
    order = np.argsort(y_prob)[::-1]
    top_k = order[:k]
    return (y_true.iloc[top_k] == "yes").mean()


def lift_at_k(y_true, y_prob, k=500):
    precision_k = precision_at_k(y_true, y_prob, k)
    base_rate = (y_true == "yes").mean()
    return precision_k / base_rate


def top_k_table(X_test, y_test, y_prob, k=500):
    result = X_test.copy()
    result["actual_y"] = y_test.values
    result["predicted_probability"] = y_prob
    result = result.sort_values("predicted_probability", ascending=False)
    result["rank"] = np.arange(1, len(result) + 1)
    return result.head(k)

## 9. Logistic Regression

In [179]:
logistic_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(
        max_iter=1000,
        random_state=RANDOM_STATE
    ))
])

logistic_model.fit(X_train, y_train)

logistic_results = evaluate_model(
    "Logistic Regression",
    logistic_model,
    X_holdout,
    y_holdout
)

print(logistic_results)

{'model': 'Logistic Regression', 'accuracy': 0.7246904588492352, 'precision': 0.5944444444444444, 'recall': 0.33700787401574805, 'f1': 0.4301507537688442, 'roc_auc': np.float64(0.7483210713525674)}


## 10. Random Forest — Bagging

In [180]:
rf_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", RandomForestClassifier(
        n_estimators=200,
        random_state=RANDOM_STATE,
        n_jobs=-1
    ))
])

rf_model.fit(X_train, y_train)

rf_results = evaluate_model(
    "Random Forest",
    rf_model,
    X_holdout,
    y_holdout
)

print(rf_results)

{'model': 'Random Forest', 'accuracy': 0.6927652342801651, 'precision': 0.6216216216216216, 'recall': 0.00905511811023622, 'f1': 0.01785021342646488, 'roc_auc': np.float64(0.723949313614668)}


## 11. Gradient Boosting

In [181]:
boosting_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", GradientBoostingClassifier(
        n_estimators=100,
        learning_rate=0.05,
        max_depth=3,
        random_state=RANDOM_STATE
    ))
])

boosting_model.fit(X_train, y_train)

boosting_results = evaluate_model(
    "Gradient Boosting",
    boosting_model,
    X_holdout,
    y_holdout
)

print(boosting_results)

{'model': 'Gradient Boosting', 'accuracy': 0.6921582908472931, 'precision': 0.5062111801242236, 'recall': 0.0641732283464567, 'f1': 0.11390635918937805, 'roc_auc': np.float64(0.6599738338911567)}


## 12. Linear Discriminant Analysis (LDA)

In [182]:
lda_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LinearDiscriminantAnalysis())
])

lda_model.fit(X_train, y_train)

lda_results = evaluate_model(
    "LDA",
    lda_model,
    X_holdout,
    y_holdout
)

print(lda_results)

{'model': 'LDA', 'accuracy': 0.725782957028405, 'precision': 0.5741424802110818, 'recall': 0.4283464566929134, 'f1': 0.4906426155580609, 'roc_auc': np.float64(0.6225705662713537)}


## 13. Quadratic Discriminant Analysis (QDA)

In [183]:
qda_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", QuadraticDiscriminantAnalysis(reg_param=0.1))
])

qda_model.fit(X_train, y_train)

qda_results = evaluate_model(
    "QDA",
    qda_model,
    X_holdout,
    y_holdout
)

print(qda_results)

{'model': 'QDA', 'accuracy': 0.7306385044913815, 'precision': 0.5793376173999011, 'recall': 0.46141732283464565, 'f1': 0.5136971290817445, 'roc_auc': np.float64(0.6891083831044461)}


## 14. Initial Model Comparison

In [184]:
results = pd.DataFrame([
    logistic_results,
    rf_results,
    boosting_results,
    lda_results,
    qda_results
])

display(results.sort_values("roc_auc", ascending=False))

,model,accuracy,precision,recall,f1,roc_auc
0,Logistic Regression,0.724690,0.594444,0.337008,0.430151,0.748321
1,Random Forest,0.692765,0.621622,0.009055,0.017850,0.723949
4,QDA,0.730639,0.579338,0.461417,0.513697,0.689108
2,Gradient Boosting,0.692158,0.506211,0.064173,0.113906,0.659974
3,LDA,0.725783,0.574142,0.428346,0.490643,0.622571


## 15. Confusion Matrices

In [185]:
models = {
    "Logistic Regression": logistic_model,
    "Random Forest": rf_model,
    "Gradient Boosting": boosting_model,
    "LDA": lda_model,
    "QDA": qda_model
}

for name, model in models.items():
    y_pred = model.predict(X_holdout)

    cm = confusion_matrix(
        (y_holdout == "yes").astype(int),
        (y_pred == "yes").astype(int)
    )

    print(name)
    print(cm)
    print()

Logistic Regression
[[5114  584]
 [1684  856]]

Random Forest
[[5684   14]
 [2517   23]]

Gradient Boosting
[[5539  159]
 [2377  163]]

LDA
[[4891  807]
 [1452 1088]]

QDA
[[4847  851]
 [1368 1172]]



## 16. Imbalance Experiment

The positive class is much smaller than the negative class. We test class weighting rather than assuming it will automatically improve the model.

In [186]:
weighted_logistic = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=RANDOM_STATE
    ))
])

weighted_logistic.fit(X_train, y_train)

weighted_logistic_results = evaluate_model(
    "Weighted Logistic Regression",
    weighted_logistic,
    X_holdout,
    y_holdout
)

print(weighted_logistic_results)

{'model': 'Weighted Logistic Regression', 'accuracy': 0.4475600873998543, 'precision': 0.35224099926524616, 'recall': 0.9437007874015748, 'f1': 0.5130016051364366, 'roc_auc': np.float64(0.7409018359805761)}


## 17. Time-Aware Tuning

Only the training period is used for tuning. `TimeSeriesSplit` keeps validation folds in temporal order.

In [187]:
tuning_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(
        max_iter=1000,
        random_state=RANDOM_STATE
    ))
])

param_grid = {
    "model__C": [0.1, 1.0, 10.0]
}

time_split = TimeSeriesSplit(n_splits=3)

grid_search = GridSearchCV(
    tuning_pipeline,
    param_grid=param_grid,
    scoring="roc_auc",
    cv=time_split,
    n_jobs=-1
)

grid_search.fit(X_train, y_train)

print("Best parameters:", grid_search.best_params_)
print("Best CV ROC-AUC:", grid_search.best_score_)

Best parameters: {'model__C': 10.0}
Best CV ROC-AUC: 0.4937521073192916


## 18. Evaluate Tuned Model on Final Holdout

In [188]:
tuned_model = grid_search.best_estimator_

tuned_results = evaluate_model(
    "Tuned Logistic Regression",
    tuned_model,
    X_holdout,
    y_holdout
)

print(tuned_results)

{'model': 'Tuned Logistic Regression', 'accuracy': 0.724933236222384, 'precision': 0.5930706521739131, 'recall': 0.3437007874015748, 'f1': 0.43519441674975073, 'roc_auc': np.float64(0.7485038264565824)}


## 19. Precision@500 and Lift@500

In [189]:
ranking_models = {
    "Logistic Regression": logistic_model,
    "Random Forest": rf_model,
    "Gradient Boosting": boosting_model,
    "LDA": lda_model,
    "QDA": qda_model,
    "Weighted Logistic Regression": weighted_logistic,
    "Tuned Logistic Regression": tuned_model
}

ranking_results = []

for name, model in ranking_models.items():
    probability = model.predict_proba(X_holdout)[:, 1]

    ranking_results.append({
        "model": name,
        "precision_at_500": precision_at_k(y_holdout, probability, 500),
        "lift_at_500": lift_at_k(y_holdout, probability, 500)
    })

ranking_results = pd.DataFrame(ranking_results)

display(
    ranking_results.sort_values("precision_at_500", ascending=False)
)

,model,precision_at_500,lift_at_500
4,QDA,0.658,2.134096
1,Random Forest,0.628,2.036797
5,Weighted Logistic Regression,0.552,1.790306
0,Logistic Regression,0.538,1.744899
6,Tuned Logistic Regression,0.534,1.731926
3,LDA,0.500,1.621654
2,Gradient Boosting,0.472,1.530841


## 20. Random-Ranking / Base-Rate Reference

In [190]:
base_rate = (y_holdout == "yes").mean()

print("Holdout subscription rate:", base_rate)
print("Expected Precision@500 under random ranking:", base_rate)

Holdout subscription rate: 0.3083272638990046
Expected Precision@500 under random ranking: 0.3083272638990046


## 21. Final Model Comparison

In [191]:
all_results = pd.concat([
    results,
    pd.DataFrame([weighted_logistic_results, tuned_results])
], ignore_index=True)

final_comparison = all_results.merge(
    ranking_results,
    on="model",
    how="left"
)

display(final_comparison.sort_values(
    "precision_at_500",
    ascending=False
))

,model,accuracy,precision,recall,f1,roc_auc,precision_at_500,lift_at_500
4,QDA,0.730639,0.579338,0.461417,0.513697,0.689108,0.658,2.134096
1,Random Forest,0.692765,0.621622,0.009055,0.017850,0.723949,0.628,2.036797
5,Weighted Logistic Regression,0.447560,0.352241,0.943701,0.513002,0.740902,0.552,1.790306
0,Logistic Regression,0.724690,0.594444,0.337008,0.430151,0.748321,0.538,1.744899
6,Tuned Logistic Regression,0.724933,0.593071,0.343701,0.435194,0.748504,0.534,1.731926
3,LDA,0.725783,0.574142,0.428346,0.490643,0.622571,0.500,1.621654
2,Gradient Boosting,0.692158,0.506211,0.064173,0.113906,0.659974,0.472,1.530841


In [192]:
# Check probability ties at the top-500 cutoff for each model

for name, model in ranking_models.items():
    probability = model.predict_proba(X_holdout)[:, 1]

    sorted_prob = np.sort(probability)[::-1]
    threshold_500 = sorted_prob[499]

    n_equal = np.sum(probability == threshold_500)

    print(
        name,
        "| 500th probability:", round(threshold_500, 6),
        "| customers tied at cutoff:", n_equal
    )

Logistic Regression | 500th probability: 0.734362 | customers tied at cutoff: 1
Random Forest | 500th probability: 0.376667 | customers tied at cutoff: 1
Gradient Boosting | 500th probability: 0.399768 | customers tied at cutoff: 1
LDA | 500th probability: 0.994563 | customers tied at cutoff: 1
QDA | 500th probability: 1.0 | customers tied at cutoff: 1341
Weighted Logistic Regression | 500th probability: 0.967702 | customers tied at cutoff: 1
Tuned Logistic Regression | 500th probability: 0.745383 | customers tied at cutoff: 1


### Ranking Tie Analysis

QDA has a severe probability-tie issue at the top-500 cutoff. The 500th-ranked customer has a predicted probability of 1.0, and 1,341 holdout customers have the same probability.

Because the bank can contact only 500 customers, selecting 500 customers from this large tied group does not provide a robust ranking.

The other evaluated models have only one customer tied at the top-500 cutoff. Therefore, Random Forest provides a more defensible ranking for the final campaign despite QDA having a slightly higher raw Precision@500.

## 22. Model Selection and Top-500 Holdout Table



## Model Selection

The final model is selected using the business objective of ranking customers for a campaign limited to 500 contacts.

Precision@500 and Lift@500 are therefore given particular importance because they directly measure the concentration of actual subscribers among the highest-ranked customers.

QDA initially achieved the highest Precision@500. However, further investigation showed that QDA assigned a predicted probability of exactly 1.0 to 1,341 holdout customers. Since the 500th-ranked customer also had a probability of 1.0, the top-500 group was selected from a large group of tied customers.

This makes the exact composition and Precision@500 of the selected 500 less robust as a ranking result.

Random Forest achieved a slightly lower Precision@500 but had only one customer tied at the top-500 cutoff. Therefore, Random Forest was selected as the final model because its ranking is more defensible for the fixed 500-customer campaign.

In [193]:
selected_model_name = "Random Forest"
selected_model = ranking_models[selected_model_name]

selected_probability = selected_model.predict_proba(X_holdout)[:, 1]

top500 = top_k_table(
    X_holdout,
    y_holdout,
    selected_probability,
    k=500
)

print("Selected model:", selected_model_name)

print("Precision@500:", precision_at_k(
    y_holdout,
    selected_probability,
    500
))

print("Lift@500:", lift_at_k(
    y_holdout,
    selected_probability,
    500
))

display(top500.head(20))

Selected model: Random Forest
Precision@500: 0.628
Lift@500: 2.0367968503937006


,age,job,marital,education,default,housing,loan,contact,month,day_of_week,...,previous,poutcome,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,actual_y,predicted_probability,rank
34648,31,technician,single,basic.9y,no,yes,no,cellular,may,thu,...,1,success,-1.8,92.893,-46.2,1.266,5099.1,yes,0.685000,1
35639,32,admin.,married,high.school,no,yes,no,cellular,may,mon,...,0,nonexistent,-1.8,92.893,-46.2,1.244,5099.1,no,0.670000,2
35557,21,blue-collar,single,basic.9y,no,yes,no,cellular,may,mon,...,0,nonexistent,-1.8,92.893,-46.2,1.244,5099.1,no,0.640131,3
32957,29,blue-collar,married,basic.9y,no,yes,no,cellular,may,mon,...,0,nonexistent,-1.8,92.893,-46.2,1.299,5099.1,no,0.640000,4
40954,48,technician,married,professional.course,no,no,no,cellular,oct,thu,...,3,success,-1.1,94.601,-49.5,0.987,4963.6,yes,0.590714,5
36056,31,technician,single,university.degree,no,yes,yes,cellular,may,tue,...,1,success,-1.8,92.893,-46.2,1.266,5099.1,no,0.590000,6
41010,33,admin.,single,university.degree,no,yes,no,cellular,oct,thu,...,3,success,-1.1,94.601,-49.5,1.025,4963.6,yes,0.585000,7
33731,34,blue-collar,single,high.school,no,yes,no,cellular,may,wed,...,1,success,-1.8,92.893,-46.2,1.281,5099.1,no,0.580000,8
40891,29,technician,married,professional.course,no,yes,no,cellular,oct,fri,...,2,success,-1.1,94.601,-49.5,0.972,4963.6,yes,0.575000,9
38283,28,technician,single,high.school,no,unknown,unknown,cellular,oct,wed,...,1,success,-3.4,92.431,-26.9,0.742,5017.5,yes,0.573333,10


## 23. Error Analysis

In [194]:
holdout_analysis = X_holdout.copy()
holdout_analysis["actual_y"] = y_holdout.values
holdout_analysis["predicted_y"] = selected_model.predict(X_holdout)
holdout_analysis["predicted_probability"] = selected_probability

false_positives = holdout_analysis[
    (holdout_analysis["actual_y"] == "no") &
    (holdout_analysis["predicted_y"] == "yes")
]

false_negatives = holdout_analysis[
    (holdout_analysis["actual_y"] == "yes") &
    (holdout_analysis["predicted_y"] == "no")
]

print("False positives:", len(false_positives))
print("False negatives:", len(false_negatives))

display(false_positives.head(10))
display(false_negatives.head(10))

False positives: 14
False negatives: 2517


,age,job,marital,education,default,housing,loan,contact,month,day_of_week,...,previous,poutcome,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,actual_y,predicted_y,predicted_probability
32957,29,blue-collar,married,basic.9y,no,yes,no,cellular,may,mon,...,0,nonexistent,-1.8,92.893,-46.2,1.299,5099.1,no,yes,0.640000
33731,34,blue-collar,single,high.school,no,yes,no,cellular,may,wed,...,1,success,-1.8,92.893,-46.2,1.281,5099.1,no,yes,0.580000
34121,28,technician,single,professional.course,no,yes,no,cellular,may,wed,...,1,success,-1.8,92.893,-46.2,1.281,5099.1,no,yes,0.560000
34329,26,admin.,single,high.school,unknown,no,no,cellular,may,thu,...,0,nonexistent,-1.8,92.893,-46.2,1.266,5099.1,no,yes,0.515000
34588,27,technician,single,professional.course,no,no,no,cellular,may,thu,...,1,success,-1.8,92.893,-46.2,1.266,5099.1,no,yes,0.535000
34666,37,technician,single,professional.course,no,yes,no,cellular,may,thu,...,1,failure,-1.8,92.893,-46.2,1.266,5099.1,no,yes,0.555000
35527,24,blue-collar,single,basic.9y,no,yes,no,cellular,may,mon,...,0,nonexistent,-1.8,92.893,-46.2,1.244,5099.1,no,yes,0.505220
35557,21,blue-collar,single,basic.9y,no,yes,no,cellular,may,mon,...,0,nonexistent,-1.8,92.893,-46.2,1.244,5099.1,no,yes,0.640131
35638,21,blue-collar,single,basic.9y,no,yes,no,cellular,may,mon,...,0,nonexistent,-1.8,92.893,-46.2,1.244,5099.1,no,yes,0.506798
35639,32,admin.,married,high.school,no,yes,no,cellular,may,mon,...,0,nonexistent,-1.8,92.893,-46.2,1.244,5099.1,no,yes,0.670000


,age,job,marital,education,default,housing,loan,contact,month,day_of_week,...,previous,poutcome,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,actual_y,predicted_y,predicted_probability
32988,39,blue-collar,married,basic.6y,unknown,no,no,cellular,may,mon,...,0,nonexistent,-1.8,92.893,-46.2,1.299,5099.1,yes,no,0.000
33035,27,blue-collar,married,basic.4y,unknown,no,no,cellular,may,mon,...,0,nonexistent,-1.8,92.893,-46.2,1.299,5099.1,yes,no,0.070
33046,33,blue-collar,married,basic.4y,no,no,no,cellular,may,mon,...,0,nonexistent,-1.8,92.893,-46.2,1.299,5099.1,yes,no,0.010
33060,47,admin.,single,high.school,no,yes,no,cellular,may,mon,...,0,nonexistent,-1.8,92.893,-46.2,1.299,5099.1,yes,no,0.050
33080,38,services,single,basic.6y,no,yes,no,cellular,may,tue,...,0,nonexistent,-1.8,92.893,-46.2,1.291,5099.1,yes,no,0.085
33124,33,blue-collar,single,basic.6y,no,yes,no,cellular,may,tue,...,1,failure,-1.8,92.893,-46.2,1.291,5099.1,yes,no,0.050
33129,32,management,single,professional.course,no,no,no,cellular,may,tue,...,0,nonexistent,-1.8,92.893,-46.2,1.291,5099.1,yes,no,0.040
33152,34,technician,single,university.degree,no,no,no,cellular,may,tue,...,0,nonexistent,-1.8,92.893,-46.2,1.291,5099.1,yes,no,0.070
33154,38,services,single,basic.9y,no,no,yes,cellular,may,tue,...,1,failure,-1.8,92.893,-46.2,1.291,5099.1,yes,no,0.040
33172,35,technician,married,basic.9y,no,no,no,cellular,may,tue,...,0,nonexistent,-1.8,92.893,-46.2,1.291,5099.1,yes,no,0.060


## 24. Save Key Results

In [195]:
os.makedirs("../results", exist_ok=True)

final_comparison.to_csv(
    "../results/model_comparison.csv",
    index=False
)

top500.to_csv(
    "../results/top500_holdout.csv",
    index=False
)

print("Saved model comparison and top-500 holdout results.")

Saved model comparison and top-500 holdout results.


## Final Findings

1. The target variable is imbalanced, and therefore multiple evaluation metrics are used rather than relying on accuracy alone.

2. The `duration` feature was excluded because it is only available after the current call and would cause information leakage in a pre-call prediction setting.

3. A temporal 80/20 split was used so that earlier observations are used for training and later observations are reserved for final evaluation.

4. The temporal holdout has a substantially higher subscription rate than the training period, indicating strong temporal target distribution shift.

5. Multiple model families were evaluated, including Logistic Regression, Random Forest, Gradient Boosting, LDA, and QDA, along with a class-weighted Logistic Regression experiment.

6. QDA achieved the highest raw Precision@500, but it produced a large probability tie at the top of the ranking. 1,341 holdout customers received a predicted probability of exactly 1.0.

7. Random Forest was therefore selected as the final model because it provided strong Precision@500 and Lift@500 while producing a much more stable ranking at the top-500 cutoff.

8. The final model should be evaluated primarily using the fixed 500-customer campaign metrics because the bank can contact only 500 customers.

9. Random Forest has low recall at the default classification threshold, but its ranking performance is strong. This highlights the distinction between threshold-based classification metrics and the top-500 ranking objective used for the bank's campaign.